# 1. Preparation

## 1.1 Imports & Configuration

In [90]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from time import perf_counter
import gc
from tqdm.auto import tqdm
import sys
from tqdm import tqdm as tqdm_text

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

import torch

from sentence_transformers import SentenceTransformer

In [91]:
SEED = 42
VAL_SIZE = 0.2
DATA_DIR = Path("data")
DENSE_MODEL_NAME = "intfloat/multilingual-e5-small"
DENSE_DIR = Path("artifacts/dense")
DENSE_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 1024

In [ ]:
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 50)

## 1.2 Data Loading

In [ ]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

In [ ]:
print(f"Train:{train.shape}")
print(f"Benchmark queries:{queries.shape}")
print(f"Benchmark items:{items.shape}")

# 2. EDA

## 2.1 Data Overview

In [ ]:
def overview(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_%": (df.isna().mean() * 100).round(4),
        "n_unique": df.nunique(dropna=True),
    })


display(overview(train))
display(overview(queries))
display(overview(items))

- Основные пропуски связаны с рейтингом и количеством отзывов.
- В каждом столбце координат пропущено 0,0008% значений в train и 0,0005% в items. В items также отсутствует 0,0185% описаний.
- Цена и координаты имеют тип object - перед числовой обработкой нужно проверить их формат.
- В queries все идентификаторы и тексты запросов уникальны, в items уникальны все item_id.

In [ ]:
display(train.describe())
display(queries.describe())
display(items.describe())

- Рейтинги объявлений в основном высокие: медиана равна 5 и в train, и в items.
- Количество отзывов распределено неравномерно: среднее заметно выше медианы, а максимумы достигают 10 678 и 18 291 соответственно.
- Поиск с доставкой в train встречается крайне редко, а в queries отсутствует.

In [ ]:
for name, df in [("train", train), ("queries", queries), ("items", items)]:
    print(f"\n{name}: память по столбцам, МБ")
    display(
        (df.memory_usage(deep=True) / 1024**2)
        .sort_values(ascending=False)
        .round(2)
    )

Основную часть памяти занимают описания и текстовые параметры объявлений.
Уменьшение числовых типов даст ограниченную экономию, поэтому оптимизацию памяти проводить не будем.

In [ ]:
display(train.head(3))
display(queries.head(3))
display(items.head(3))

In [ ]:
print("Full duplicates:")
print(f"train:{train.duplicated().sum()}")
print(f"queries:{queries.duplicated().sum()}")
print(f"items:{items.duplicated().sum()}")

Повторы в обучающей выборке могут отражать повторные взаимодействия пользователей, поэтому перед удалением нужно решить, будем ли учитывать их частоту.
В queries и items полных дубликатов нет.

## 2.2 Query Statistics

In [ ]:
QUERY_COLS = [
    col for col in queries.columns
    if col.startswith("search_")
]

print("Unique query texts:")
print("train:", train["search_query"].nunique())
print("benchmark:", queries["search_query"].nunique())

query_lengths = (
    queries["search_query"]
    .fillna("")
    .str.split()
    .str.len()
)

display(query_lengths.describe())

In [ ]:
plt.figure(figsize=(8, 4))

query_lengths.clip(upper=15).hist(
    bins=np.arange(0, 17) - 0.5
)

plt.xlabel("Number of words")
plt.ylabel("Number of queries")
plt.title("Search Query Length")
plt.show()

In [ ]:
relevant_per_query = (
    train
    .groupby(QUERY_COLS, dropna=False)["item_id"]
    .nunique()
)

display(
    relevant_per_query.describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

print(
    "Queries with multiple relevant items:",
    f"{(relevant_per_query > 1).mean():.2%}"
)

- Запросы в benchmark короткие: медианная длина - 3 слова. Поэтому стоит сравнить word- и character-level TF-IDF, проверив, помогают ли символьные n-граммы учитывать различия в написании.
- Для 85% поисковых контекстов в train наблюдается только одно положительное объявление, для 15% — несколько.
- При расчёте `Recall@50` будем учитывать все уникальные положительные `item_id` для каждого контекста;
- отсутствие взаимодействия само по себе не означает нерелевантность объявления.

In [ ]:
normalize_query = lambda s: (
    s.fillna("")
    .str.lower()
    .str.strip()
)

train_query_set = set(
    normalize_query(train["search_query"])
)

benchmark_query_norm = normalize_query(
    queries["search_query"]
)

query_overlap = benchmark_query_norm.isin(
    train_query_set
).mean()

print(
    f"Benchmark queries seen in train: "
    f"{query_overlap:.2%}"
)

In [ ]:
train_item_ids = set(train["item_id"].astype(str))
benchmark_item_ids = set(items["item_id"].astype(str))

item_overlap = (
    len(train_item_ids & benchmark_item_ids)
    / len(train_item_ids)
)

print(
    f"Train items present in benchmark corpus: "
    f"{item_overlap:.2%}"
)

63,01% текстов benchmark-запросов не встречались в train.
Случайное разбиение по строкам может завысить локальную метрику, если одинаковые формулировки окажутся в обеих частях.
Поэтому валидация должна включать отдельную оценку на unseen queries.

## 2.4 Query–Item Examples

In [ ]:
cols = [
    "search_query",
    "search_infm_params_text",
    "search_category",
    "search_location_id",
    "item_title_raw",
    "item_infm_params_text",
    "item_category_id",
    "item_microcat_id",
    "item_location_id",
]

cols = [col for col in cols if col in train.columns]

display(
    train[cols]
    .sample(15, random_state=SEED)
)

В положительных парах текст запроса не всегда буквально совпадает с заголовком объявления.

# 3. Validation Strategy

Основная локальная проверка - поиск для новых формулировок запросов. Выделим в validation 20% уникальных нормализованных текстов. Все взаимодействия с одной формулировкой попадут в одну часть независимо от локации и параметров.

Для расчёта Recall@50 отдельный запрос определяется полным поисковым контекстом QUERY_COLS. Корпус поиска зафиксируем до экспериментов.

Разбиение на train и validation

In [ ]:
query_norm = normalize_query(train["search_query"])
unique_queries = np.sort(query_norm.unique())

rng = np.random.default_rng(SEED)
n_val = int(np.ceil(len(unique_queries) * VAL_SIZE))

val_texts = rng.choice(
    unique_queries,
    size=n_val,
    replace=False,
)

is_val = query_norm.isin(val_texts)

train_fit = train.loc[~is_val].copy()
val_pairs = train.loc[is_val, QUERY_COLS + ["item_id"]].copy()

assert set(query_norm[~is_val]).isdisjoint(query_norm[is_val])

print(f"Train query texts: {query_norm[~is_val].nunique():,}")
print(f"Validation query texts: {query_norm[is_val].nunique():,}")
print(f"Train interactions: {len(train_fit):,}")
print(f"Validation interactions: {len(val_pairs):,}")

Подготовим данные для локальной валидации: присвоим отдельный query_id каждой уникальной комбинации признаков запроса и соберём для неё множество релевантных объявлений без повторов.

In [ ]:
val_pairs["query_id"] = (
    val_pairs
    .groupby(QUERY_COLS, dropna=False, sort=False)
    .ngroup()
)

val_queries = (
    val_pairs[["query_id"] + QUERY_COLS]
    .drop_duplicates("query_id")
    .reset_index(drop=True)
)

val_relevant = (
    val_pairs
    .groupby("query_id")["item_id"]
    .agg(set)
    .to_dict()
)

assert val_queries["query_id"].is_unique
assert set(val_queries["query_id"]) == set(val_relevant)

print(f"Validation query contexts: {len(val_queries):,}")
print(
    "Unique positive query-item pairs:",
    f"{sum(map(len, val_relevant.values())):,}",
)

Корпус для локального поиска:

1. Соберём список объявлений, среди которых будем искать ответы на валидационные запросы.
2. Уберём повторы по item_id и проверим, что все правильные ответы есть в этом списке.

In [ ]:
ITEM_COLS = [
    col for col in train.columns
    if col.startswith("item_")
]

local_items = (
    train[ITEM_COLS]
    .drop_duplicates("item_id")
    .reset_index(drop=True)
)

corpus_ids = set(local_items["item_id"])

assert all(
    relevant_ids.issubset(corpus_ids)
    for relevant_ids in val_relevant.values()
)

print(f"Local corpus size: {len(local_items):,}")

Метрика Recall@50

In [ ]:
def recall_at_k(predictions, relevant, k=50):
    if k < 1:
        raise ValueError("k must be positive")

    if set(predictions) != set(relevant):
        raise ValueError(
            "Predictions must contain every validation query exactly once"
        )

    scores = []

    for query_id, relevant_ids in relevant.items():
        retrieved_ids = set(predictions[query_id][:k])

        score = len(retrieved_ids & relevant_ids) / len(relevant_ids)
        scores.append(score)

    return float(np.mean(scores))

Локальная метрика будет показывать качество на unseen queries. Она не оценивает преимущество методов, использующих историю знакомых запросов.

Все модели сравниваем на одинаковых запросах и корпусе.
Обучение на взаимодействиях и расчёт статистик релевантности выполняем только по train_fit;
тексты объявлений из validation доступны для индексации.

# 4. Sparse Retrieval

## 4.1 Word TF-IDF Baseline

Начнём с поиска по сходству текста запроса и заголовка объявления.
Используем TF-IDF по отдельным словам и косинусное сходство.
Результат станет точкой отсчёта для следующих экспериментов.

1. Подготовка текста

In [ ]:
def prepare_text(series):
    return (
        series
        .fillna("")
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


item_titles = prepare_text(local_items["item_title_raw"])
query_texts = prepare_text(val_queries["search_query"])

2. TF-IDF

Словарь и IDF рассчитываем по локальному корпусу объявлений. Запросы преобразуем тем же векторизатором.

In [ ]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 1),
    lowercase=False,
    norm="l2",
    dtype=np.float32,
)

started = perf_counter()

item_vectors = word_vectorizer.fit_transform(item_titles)
query_vectors = word_vectorizer.transform(query_texts)

index_time = perf_counter() - started

print(f"Vocabulary size: {len(word_vectorizer.vocabulary_):,}")
print(f"Item matrix: {item_vectors.shape}")
print(f"Query matrix: {query_vectors.shape}")
print(f"Vectorization time: {index_time:.2f} s")

3. Проверка покрытия запросов

In [ ]:
zero_query_mask = query_vectors.getnnz(axis=1) == 0
zero_query_share = zero_query_mask.mean()

print(f"Queries with zero vectors: {zero_query_share:.2%}")

if zero_query_mask.any():
    display(
        val_queries.loc[zero_query_mask, ["search_query"]]
        .drop_duplicates()
        .head(10)
    )

Нулевой вектор означает, что после токенизации в запросе не осталось слов из словаря заголовков.
Такие запросы этот baseline не покрывает.
Долю нулевых векторов будем сравнивать с Char TF-IDF, чтобы проверить, расширяют ли символьные n-граммы покрытие.

4. Поиск top-50

При L2-нормировке скалярное произведение TF-IDF-векторов равно косинусному сходству. Считаем его батчами и сохраняем разреженное представление.

Возвращаем до 50 кандидатов с положительным сходством. Для нулевого запроса выдача будет пустой.

In [ ]:
def retrieve_top_k(
    query_vectors,
    item_vectors,
    query_ids,
    item_ids,
    k=50,
    batch_size=64,
):
    item_matrix_t = item_vectors.T.tocsc()
    predictions = {}

    for start in range(0, query_vectors.shape[0], batch_size):
        stop = min(start + batch_size, query_vectors.shape[0])

        similarities = (
            query_vectors[start:stop] @ item_matrix_t
        ).tocsr()

        similarities.sum_duplicates()
        similarities.eliminate_zeros()

        for row_idx in range(stop - start):
            left = similarities.indptr[row_idx]
            right = similarities.indptr[row_idx + 1]

            candidate_indices = similarities.indices[left:right]
            scores = similarities.data[left:right]

            if len(scores) > k:
                selected = np.argpartition(scores, -k)[-k:]
            else:
                selected = np.arange(len(scores))

            # Сортируем выбранных кандидатов по убыванию сходства.
            selected = selected[
                np.argsort(-scores[selected], kind="stable")
            ]

            query_id = query_ids[start + row_idx]
            predictions[query_id] = (
                item_ids[candidate_indices[selected]].tolist()
            )

    return predictions

In [ ]:
started = perf_counter()

baseline_predictions = retrieve_top_k(
    query_vectors=query_vectors,
    item_vectors=item_vectors,
    query_ids=val_queries["query_id"].to_numpy(),
    item_ids=local_items["item_id"].to_numpy(),
)

search_time = perf_counter() - started

baseline_recall = recall_at_k(
    baseline_predictions,
    val_relevant,
    k=50,
)

print(f"Recall@50: {baseline_recall:.4f}")
print(f"Search time: {search_time:.2f} s")

In [ ]:
experiments = pd.DataFrame([
    {
        "model": "Word TF-IDF",
        "item_fields": "title",
        "recall@50": baseline_recall,
        "zero_queries_%": zero_query_share * 100,
        "vectorization_s": index_time,
        "search_s": search_time,
    }
])

display(experiments.round(4))

Word TF-IDF по заголовкам дал результат в 14,35%.
Нулевые векторы встречаются только у 0,30% запросов, поэтому отсутствие слов в словаре не объясняет основную часть ошибок.


## 4.2 Character TF-IDF !!!У меня не потянуло!!!

Проверим, помогает ли поиск по символьным n-граммам учитывать различия в написании и формах слов.
Используем char_wb с длиной n-грамм от 3 до 5 символов.

1. Построение символьного TF-IDF

In [ ]:
# char_vectorizer = TfidfVectorizer(
#     analyzer="char_wb",
#     ngram_range=(3, 5),
#     lowercase=False,
#     norm="l2",
#     dtype=np.float32,
# )
#
# started = perf_counter()
#
# char_item_vectors = char_vectorizer.fit_transform(item_titles)
# char_query_vectors = char_vectorizer.transform(query_texts)
#
# char_index_time = perf_counter() - started
#
# print(f"Vocabulary size: {len(char_vectorizer.vocabulary_):,}")
# print(f"Item matrix: {char_item_vectors.shape}")
# print(f"Query matrix: {char_query_vectors.shape}")
# print(f"Vectorization time: {char_index_time:.2f} s")

2. Проверка покрытия запросов

In [ ]:
# char_zero_mask = char_query_vectors.getnnz(axis=1) == 0
# char_zero_share = char_zero_mask.mean()
#
# print(f"Word TF-IDF zero queries: {zero_query_share:.2%}")
# print(f"Char TF-IDF zero queries: {char_zero_share:.2%}")
#
# covered_by_char = zero_query_mask & ~char_zero_mask
#
# print(
#     "Previously zero queries now covered:",
#     f"{covered_by_char.sum():,}",
# )
#
# if covered_by_char.any():
#     display(
#         val_queries.loc[covered_by_char, ["search_query"]]
#         .drop_duplicates()
#         .head(10)
#     )

Вполне ожидаемо n-граммы устранили проблему с нулевыми векторами.

3. Поиск и оценка

In [ ]:
# started = perf_counter()
#
# char_predictions = retrieve_top_k(
#     query_vectors=char_query_vectors,
#     item_vectors=char_item_vectors,
#     query_ids=val_queries["query_id"].to_numpy(),
#     item_ids=local_items["item_id"].to_numpy(),
#     batch_size=16,
# )
#
# char_search_time = perf_counter() - started
#
# char_recall = recall_at_k(
#     char_predictions,
#     val_relevant,
#     k=50,
# )
#
# print(f"Char Recall@50: {char_recall:.4f}")
# print(f"Delta vs Word: {char_recall - baseline_recall:+.4f}")
# print(f"Search time: {char_search_time:.2f} s")

4. Сравнение экспериментов

In [ ]:
# experiments = pd.DataFrame([
#     {
#         "model": "Word TF-IDF",
#         "item_fields": "title",
#         "recall@50": baseline_recall,
#         "delta_vs_word": 0.0,
#         "zero_queries_%": zero_query_share * 100,
#         "vectorization_s": index_time,
#         "search_s": search_time,
#     },
#     {
#         "model": "Char TF-IDF",
#         "item_fields": "title",
#         "recall@50": char_recall,
#         "delta_vs_word": char_recall - baseline_recall,
#         "zero_queries_%": char_zero_share * 100,
#         "vectorization_s": char_index_time,
#         "search_s": char_search_time,
#     },
# ])
#
# display(experiments.round(4))

Поиск top-50 в текущей реализации оказался слишком ресурсоёмким для доступного оборудования, поэтому эксперимент остановлен без оценки Recall@50. Гипотеза об улучшении качества остаётся непроверенной.

## 4.3 BM25

Проверим BM25 на тех же заголовках. В отличие от TF-IDF, BM25 ограничивает вклад повторений слова и отдельно учитывает длину документа.

Начальные параметры: k1=1.5, b=0.75.

In [ ]:
BM25_K1 = 1.5
BM25_B = 0.75

1. Построение BM25

Используем разреженные матрицы, чтобы искать кандидатов той же функцией retrieve_top_k.
L2-нормировку здесь не применяем: скалярное произведение будет давать BM25 score.

In [ ]:
bm25_vectorizer = CountVectorizer(
    analyzer=word_vectorizer.build_analyzer(),
    vocabulary=word_vectorizer.vocabulary_,
    dtype=np.float32,
)

started = perf_counter()

bm25_item_vectors = bm25_vectorizer.fit_transform(item_titles)
bm25_query_vectors = bm25_vectorizer.transform(query_texts)

bm25_query_vectors.data.fill(1.0)

n_docs = bm25_item_vectors.shape[0]

doc_lengths = np.asarray(
    bm25_item_vectors.sum(axis=1)
).ravel()

avg_doc_length = doc_lengths.mean()

if avg_doc_length == 0:
    raise ValueError("The corpus contains no recognized tokens")

doc_frequency = np.asarray(
    bm25_item_vectors.getnnz(axis=0)
).ravel()

bm25_idf = np.log1p(
    (n_docs - doc_frequency + 0.5)
    / (doc_frequency + 0.5)
).astype(np.float32)

length_penalty = BM25_K1 * (
    1 - BM25_B
    + BM25_B * doc_lengths / avg_doc_length
)

denominator = bm25_item_vectors.data + np.repeat(
    length_penalty,
    np.diff(bm25_item_vectors.indptr),
)

bm25_item_vectors.data *= BM25_K1 + 1
bm25_item_vectors.data /= denominator
bm25_item_vectors.data *= bm25_idf[
    bm25_item_vectors.indices
]

del denominator

bm25_index_time = perf_counter() - started

print(f"Vocabulary size: {len(bm25_vectorizer.vocabulary_):,}")
print(f"Average title length: {avg_doc_length:.2f} tokens")
print(f"Indexing time: {bm25_index_time:.2f} s")

2. Поиск и оценка

In [ ]:
started = perf_counter()

bm25_predictions = retrieve_top_k(
    query_vectors=bm25_query_vectors,
    item_vectors=bm25_item_vectors,
    query_ids=val_queries["query_id"].to_numpy(),
    item_ids=local_items["item_id"].to_numpy(),
    batch_size=64,
)

bm25_search_time = perf_counter() - started

bm25_recall = recall_at_k(
    bm25_predictions,
    val_relevant,
    k=50,
)

print(f"BM25 Recall@50: {bm25_recall:.4f}")
print(f"Delta vs Word: {bm25_recall - baseline_recall:+.4f}")
print(f"Search time: {bm25_search_time:.2f} s")

3. Добавление результата в таблицу

In [ ]:
bm25_zero_share = (
    bm25_query_vectors.getnnz(axis=1) == 0
).mean()

bm25_result = pd.DataFrame([
    {
        "model": "BM25",
        "item_fields": "title",
        "recall@50": bm25_recall,
        "delta_vs_word": bm25_recall - baseline_recall,
        "zero_queries_%": bm25_zero_share * 100,
        "vectorization_s": bm25_index_time,
        "search_s": bm25_search_time,
    }
])

experiments = pd.concat(
    [
        experiments.loc[experiments["model"] != "BM25"],
        bm25_result,
    ],
    ignore_index=True,
)

display(
    experiments
    .sort_values("recall@50", ascending=False)
    .round(4)
)

BM25 и Word TF-IDF используют одинаковый словарь, поэтому BM25 не устраняет нулевые запросы

BM25 повысил `Recall@50` с 0.1435 до 0.1515 (+0,8 п.п.) при практически одинаковом времени поиска: 54,9 с против 54,4 с. Для следующих экспериментов с текстовыми полями выбираем BM25.

## 4.4 Text Representation

Проверим, помогает ли расширение текста объявления находить положительные кандидаты. Сравним title, title + params и title + params + description.

Метод поиска и его параметры фиксируем. Запросы оставляем без изменений: используем только search_query. Для каждого состава текста заново строим словарь и индекс.

1. Функция эксперимента с прогрессом

In [125]:
def build_bm25_vectors(item_texts, query_texts, k1, b):
    vectorizer = CountVectorizer(
        lowercase=False,
        dtype=np.float32,
    )

    item_matrix = vectorizer.fit_transform(item_texts)
    query_matrix = vectorizer.transform(query_texts)
    query_matrix.data.fill(1.0)

    doc_lengths = np.asarray(item_matrix.sum(axis=1)).ravel()
    avg_length = doc_lengths.mean()

    if avg_length == 0:
        raise ValueError("The corpus contains no recognized tokens")

    n_docs = item_matrix.shape[0]
    doc_frequency = np.asarray(
        item_matrix.getnnz(axis=0)
    ).ravel()

    idf = np.log1p(
        (n_docs - doc_frequency + 0.5)
        / (doc_frequency + 0.5)
    ).astype(np.float32)

    length_penalty = k1 * (
        1 - b + b * doc_lengths / avg_length
    )

    denominator = item_matrix.data + np.repeat(
        length_penalty,
        np.diff(item_matrix.indptr),
    )

    item_matrix.data *= k1 + 1
    item_matrix.data /= denominator
    item_matrix.data *= idf[item_matrix.indices]

    return item_matrix, query_matrix

In [ ]:
def run_bm25_experiment(columns, name, batch_size=16):
    print(f"BM25 | {name}", flush=True)
    print("1/3 Подготовка текстов...", flush=True)

    texts = local_items[columns[0]].fillna("")

    for column in columns[1:]:
        texts = texts + " " + local_items[column].fillna("")

    texts = prepare_text(texts)

    print("2/3 Построение индекса...", flush=True)
    started = perf_counter()

    item_matrix, query_matrix = build_bm25_vectors(
        item_texts=texts,
        query_texts=query_texts,
        k1=BM25_K1,
        b=BM25_B,
    )

    index_seconds = perf_counter() - started
    zero_share = (query_matrix.getnnz(axis=1) == 0).mean()

    print(f"Индекс построен за {index_seconds:.2f} с", flush=True)

    del texts
    gc.collect()

    print("3/3 Поиск top-50...", flush=True)
    started = perf_counter()

    item_matrix_t = item_matrix.T.tocsc()
    query_ids = val_queries["query_id"].to_numpy()
    item_ids = local_items["item_id"].to_numpy()

    predictions = {}
    n_queries = query_matrix.shape[0]
    k = 50

    with tqdm(
        total=n_queries,
        desc=name,
        unit="запрос",
        mininterval=1.0,
    ) as progress:
        for start in range(0, n_queries, batch_size):
            stop = min(start + batch_size, n_queries)

            similarities = (
                query_matrix[start:stop] @ item_matrix_t
            ).tocsr()

            similarities.sum_duplicates()
            similarities.eliminate_zeros()

            for row_idx in range(stop - start):
                left = similarities.indptr[row_idx]
                right = similarities.indptr[row_idx + 1]

                candidate_indices = similarities.indices[left:right]
                scores = similarities.data[left:right]

                if len(scores) > k:
                    selected = np.argpartition(scores, -k)[-k:]
                else:
                    selected = np.arange(len(scores))

                selected = selected[
                    np.argsort(-scores[selected], kind="stable")
                ]

                predictions[query_ids[start + row_idx]] = (
                    item_ids[candidate_indices[selected]].tolist()
                )

            progress.update(stop - start)
            del candidate_indices, scores, selected, similarities

    search_seconds = perf_counter() - started

    score = recall_at_k(predictions, val_relevant, k=50)

    result = {
        "model": "BM25",
        "item_fields": name,
        "recall@50": score,
        "delta_vs_word": score - baseline_recall,
        "zero_queries_%": zero_share * 100,
        "vectorization_s": index_seconds,
        "search_s": search_seconds,
    }

    print(
        f"Recall@50: {score:.4f} | "
        f"Delta vs title: {score - bm25_recall:+.4f} | "
        f"Search time: {search_seconds:.2f} s",
        flush=True,
    )

    del item_matrix, item_matrix_t, query_matrix, predictions
    gc.collect()

    return result

2. Хранение результатов

In [ ]:
if "bm25_text_results" not in globals():
    bm25_text_results = {}

bm25_text_results["title"] = {
    "model": "BM25",
    "item_fields": "title",
    "recall@50": bm25_recall,
    "delta_vs_word": bm25_recall - baseline_recall,
    "zero_queries_%": bm25_zero_share * 100,
    "vectorization_s": bm25_index_time,
    "search_s": bm25_search_time,
}

3. title + params

In [ ]:
bm25_text_results["title + params"] = run_bm25_experiment(
    columns=[
        "item_title_raw",
        "item_infm_params_text",
    ],
    name="title + params",
)

4. Сравнение экспериментов

In [ ]:
representation_results = pd.DataFrame(
    bm25_text_results.values()
)

representation_results["delta_vs_title"] = (
    representation_results["recall@50"] - bm25_recall
)

display(
    representation_results[[
        "item_fields",
        "recall@50",
        "delta_vs_title",
        "zero_queries_%",
        "vectorization_s",
        "search_s",
    ]].round(4)
)

replace_mask = (
    (experiments["model"] == "BM25")
    & experiments["item_fields"].isin(bm25_text_results)
)

experiments = pd.concat(
    [
        experiments.loc[~replace_mask],
        representation_results.drop(columns="delta_vs_title"),
    ],
    ignore_index=True,
)

Добавление параметров снизило Recall@50 с 0.1515 до 0.1437 (-0,78 п.п.), а время поиска выросло с 55 секунд до 26 минут. Простое объединение заголовка и параметров не дало улучшения.

# 5. Dense Retrieval

Проверим, помогает ли семантическое представление текста находить положительные объявления без точного совпадения слов.

Используем готовую multilingual-e5-small без дообучения. Объявления представим заголовками, запросы - текстом search_query.

## 5.1 GPU & Speed Check

In [128]:
if not torch.cuda.is_available():
    raise RuntimeError("Для этого запуска требуется CUDA GPU")

DENSE_BATCH_SIZE = 64

dense_model = SentenceTransformer(
    DENSE_MODEL_NAME,
    device="cuda",
)

dense_model.max_seq_length = 64
dense_model.half()
dense_model.eval()

dense_item_codes, dense_unique_titles = pd.factorize(
    item_titles,
    sort=False,
)

dense_query_codes, dense_unique_queries = pd.factorize(
    query_texts,
    sort=False,
)

print("GPU:", torch.cuda.get_device_name(0))
print(f"Unique item texts: {len(dense_unique_titles):,}")
print(f"Unique query texts: {len(dense_unique_queries):,}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9426.25it/s]


GPU: NVIDIA GeForce RTX 2070 SUPER
Unique item texts: 205,435
Unique query texts: 14,884


## 5.2 Embeddings

Получим нормализованные эмбеддинги заголовков и запросов. Повторяющиеся тексты кодируем один раз, сохраняя соответствие исходным строкам.

Модель используем без дообучения. Эмбеддинги и идентификаторы сохраняем на диск для повторного использования.

In [81]:
torch.cuda.synchronize()
started = perf_counter()

dense_item_embeddings = dense_model.encode(
    ["passage: " + text for text in dense_unique_titles],
    batch_size=DENSE_BATCH_SIZE,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float32)

dense_query_embeddings = dense_model.encode(
    ["query: " + text for text in dense_unique_queries],
    batch_size=DENSE_BATCH_SIZE,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype(np.float32)

# После вычисления моделью в FP16 нормируем векторы в FP32.
for embeddings in (dense_item_embeddings, dense_query_embeddings):
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)

    if not np.isfinite(embeddings).all() or (norms == 0).any():
        raise ValueError("Invalid embeddings")

    embeddings /= norms

torch.cuda.synchronize()
dense_encoding_time = perf_counter() - started

print(f"Item embeddings: {dense_item_embeddings.shape}")
print(f"Query embeddings: {dense_query_embeddings.shape}")
print(f"Encoding time: {dense_encoding_time:.2f} s")

Batches: 100%|██████████| 233/233 [00:03<00:00, 70.66it/s]


Item embeddings: (205435, 384)
Query embeddings: (14884, 384)
Encoding time: 47.65 s


In [82]:
np.savez(
    DENSE_DIR / "validation_embeddings.npz",
    item_embeddings=dense_item_embeddings,
    query_embeddings=dense_query_embeddings,
    item_codes=dense_item_codes,
    query_codes=dense_query_codes,
    item_ids=local_items["item_id"].to_numpy(dtype=str),
    query_ids=val_queries["query_id"].to_numpy(),
    model_name=np.array(DENSE_MODEL_NAME),
    max_seq_length=np.array(dense_model.max_seq_length),
    encoding_seconds=np.array(dense_encoding_time),
)

print("Embeddings saved")

Embeddings saved


## 5.3 GPU Retrieval

Для нормализованных эмбеддингов скалярное произведение равно косинусному сходству. Выполним точный поиск top-50 по всему локальному корпусу на GPU, обрабатывая запросы батчами.

Одинаковые тексты запросов получают одинаковую выдачу. При оценке восстановим все поисковые контексты validation.

Функция поиска

In [84]:
@torch.inference_mode()
def dense_top_k(
    unique_query_embeddings,
    unique_item_embeddings,
    item_codes,
    k=50,
    batch_size=128,
):
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required")

    n_queries = len(unique_query_embeddings)
    n_items = len(item_codes)

    if n_queries == 0 or n_items == 0:
        raise ValueError("Queries and corpus must be non-empty")

    k = min(k, n_items)

    # Восстанавливаем вектор каждого объявления:
    # одинаковые заголовки могут принадлежать разным item_id.
    unique_items_gpu = torch.as_tensor(
        unique_item_embeddings,
        dtype=torch.float32,
        device="cuda",
    )

    codes_gpu = torch.as_tensor(
        item_codes,
        dtype=torch.long,
        device="cuda",
    )

    items_gpu = unique_items_gpu.index_select(0, codes_gpu)

    del unique_items_gpu, codes_gpu

    queries_gpu = torch.as_tensor(
        unique_query_embeddings,
        dtype=torch.float32,
        device="cuda",
    )

    top_indices = np.empty((n_queries, k), dtype=np.int32)
    top_scores = np.empty((n_queries, k), dtype=np.float32)

    try:
        with tqdm(
            total=n_queries,
            desc="Dense retrieval",
            unit="запрос",
            mininterval=1.0,
        ) as progress:
            for start in range(0, n_queries, batch_size):
                stop = min(start + batch_size, n_queries)

                similarities = queries_gpu[start:stop] @ items_gpu.T

                scores, indices = torch.topk(
                    similarities,
                    k=k,
                    dim=1,
                    largest=True,
                    sorted=True,
                )

                top_indices[start:stop] = indices.cpu().numpy()
                top_scores[start:stop] = scores.cpu().numpy()

                del similarities, scores, indices
                progress.update(stop - start)

    finally:
        del items_gpu, queries_gpu
        torch.cuda.empty_cache()

    return top_indices, top_scores

Запуск поиска и восстановление выдачи по query_id:

In [85]:
torch.cuda.synchronize()
started = perf_counter()

dense_top_indices, dense_top_scores = dense_top_k(
    unique_query_embeddings=dense_query_embeddings,
    unique_item_embeddings=dense_item_embeddings,
    item_codes=dense_item_codes,
    k=50,
    batch_size=128,
)

item_ids = local_items["item_id"].to_numpy()
query_ids = val_queries["query_id"].to_numpy()

unique_query_predictions = item_ids[dense_top_indices].tolist()

dense_predictions = {
    query_id: unique_query_predictions[int(text_code)]
    for query_id, text_code in zip(query_ids, dense_query_codes)
}

torch.cuda.synchronize()
dense_search_time = perf_counter() - started

assert set(dense_predictions) == set(val_relevant)
assert all(
    len(candidates) == len(set(candidates)) == 50
    for candidates in dense_predictions.values()
)

print(f"Search time: {dense_search_time:.2f} s")

Dense retrieval: 100%|██████████| 14884/14884 [00:01<00:00, 9292.62запрос/s]


Search time: 1.81 s


## 5.4 Evaluation

Сравним dense-поиск с BM25 по заголовкам на том же корпусе и validation.

In [86]:
dense_recall = recall_at_k(
    dense_predictions,
    val_relevant,
    k=50,
)

print(f"Dense Recall@50: {dense_recall:.4f}")
print(f"BM25 Recall@50:  {bm25_recall:.4f}")
print(f"Delta vs BM25:   {dense_recall - bm25_recall:+.4f}")

Dense Recall@50: 0.1605
BM25 Recall@50:  0.1515
Delta vs BM25:   +0.0091


Сохраняем выдачу и добавляем результат в таблицу:

In [87]:
np.savez(
    DENSE_DIR / "validation_top50.npz",
    top_item_ids=local_items["item_id"].to_numpy(dtype=str)[
        dense_top_indices
    ],
    top_scores=dense_top_scores,
    query_codes=dense_query_codes,
    query_ids=query_ids,
    recall_at_50=np.array(dense_recall),
    search_seconds=np.array(dense_search_time),
)

dense_result = pd.DataFrame([{
    "model": "Multilingual E5-small",
    "item_fields": "title",
    "recall@50": dense_recall,
    "delta_vs_word": dense_recall - baseline_recall,
    "zero_queries_%": np.nan,
    "vectorization_s": dense_encoding_time,
    "search_s": dense_search_time,
}])

replace_mask = (
    (experiments["model"] == "Multilingual E5-small")
    & (experiments["item_fields"] == "title")
)

experiments = pd.concat(
    [experiments.loc[~replace_mask], dense_result],
    ignore_index=True,
)

display(
    experiments
    .sort_values("recall@50", ascending=False)
    .round(4)
)

,model,item_fields,recall@50,zero_queries_%,vectorization_s,search_s,delta_vs_word
3,Multilingual E5-small,title,0.1605,NaN,47.6527,1.8144,0.0171
1,BM25,title,0.1515,0.2954,2.7702,54.9301,0.0080
2,BM25,title + params,0.1437,0.2293,40.0910,1569.3237,0.0002
0,Word TF-IDF,title,0.1435,0.2954,2.4670,54.3734,NaN


1. Multilingual E5-small по заголовкам повысил `Recall@50` до 0.1605, обойдя BM25 на 0,91 п.п.
2. Получение эмбеддингов заняло 47,7 с, поиск на GPU - 1,8 с.
3. Фиксируем dense по заголовкам как лучший отдельный метод и проверяем, улучшит ли добавление параметров представление объявления.

## 5.5 Dense: Title + Params

Добавим параметры к заголовку объявления. Модель, ограничение длины, запросы и способ поиска оставим прежними.

Эмбеддинги запросов используем повторно. Результат сравним с dense по одним заголовкам.

1. Подготовка текста

In [101]:
dense_model.max_seq_length = 64
DENSE_BATCH_SIZE = 16

In [102]:
dense_params_texts = prepare_text(
    local_items["item_title_raw"].fillna("")
    + ". "
    + local_items["item_infm_params_text"].fillna("")
)

dense_params_codes, dense_unique_params = pd.factorize(
    dense_params_texts,
    sort=False,
)

print(f"Unique title + params texts: {len(dense_unique_params):,}")

del dense_params_texts
gc.collect()

Unique title + params texts: 344,597


4

2. Эмбеддинги объявлений

In [103]:
n_texts = len(dense_unique_params)
embedding_dim = dense_model.get_sentence_embedding_dimension()

dense_params_embeddings = np.empty(
    (n_texts, embedding_dim),
    dtype=np.float32,
)

torch.cuda.synchronize()
started = perf_counter()

with tqdm_text(
    total=n_texts,
    desc="Кодирование title + params",
    unit="объявл.",
    file=sys.stdout,
    mininterval=1.0,
) as progress:

    for start in range(0, n_texts, CHUNK_SIZE):
        stop = min(start + CHUNK_SIZE, n_texts)

        texts = [
            "passage: " + text
            for text in dense_unique_params[start:stop]
        ]

        vectors = dense_model.encode(
            texts,
            batch_size=DENSE_BATCH_SIZE,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        ).astype(np.float32)

        norms = np.linalg.norm(vectors, axis=1, keepdims=True)

        if not np.isfinite(vectors).all() or (norms == 0).any():
            raise ValueError(
                f"Invalid embeddings in rows {start}:{stop}"
            )

        vectors /= norms
        dense_params_embeddings[start:stop] = vectors

        progress.update(stop - start)

        del texts, vectors, norms

torch.cuda.synchronize()
dense_params_encoding_time = perf_counter() - started

print(f"Embeddings: {dense_params_embeddings.shape}")
print(f"Item encoding time: {dense_params_encoding_time:.2f} s")

C:\Users\alesh\AppData\Local\Temp\ipykernel_20256\2624519130.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = dense_model.get_sentence_embedding_dimension()


Кодирование title + params: 100%|██████████| 344597/344597 [05:09<00:00, 1111.74объявл./s]
Embeddings: (344597, 384)
Item encoding time: 309.96 s


3. Поиск и оценка

In [104]:
torch.cuda.synchronize()
started = perf_counter()

dense_params_indices, dense_params_scores = dense_top_k(
    unique_query_embeddings=dense_query_embeddings,
    unique_item_embeddings=dense_params_embeddings,
    item_codes=dense_params_codes,
    k=50,
    batch_size=128,
)

item_ids = local_items["item_id"].to_numpy()
query_ids = val_queries["query_id"].to_numpy()

params_unique_predictions = item_ids[dense_params_indices].tolist()

dense_params_predictions = {
    query_id: params_unique_predictions[int(text_code)]
    for query_id, text_code in zip(query_ids, dense_query_codes)
}

torch.cuda.synchronize()
dense_params_search_time = perf_counter() - started

assert set(dense_params_predictions) == set(val_relevant)
assert all(
    len(candidates) == len(set(candidates)) == 50
    for candidates in dense_params_predictions.values()
)

dense_params_recall = recall_at_k(
    dense_params_predictions,
    val_relevant,
    k=50,
)

print(f"Dense title Recall@50:          {dense_recall:.4f}")
print(f"Dense title + params Recall@50: {dense_params_recall:.4f}")
print(f"Delta vs dense title: {dense_params_recall - dense_recall:+.4f}")
print(f"Search time: {dense_params_search_time:.2f} s")

Dense retrieval: 100%|██████████| 14884/14884 [00:01<00:00, 10822.63запрос/s]


Dense title Recall@50:          0.1605
Dense title + params Recall@50: 0.1836
Delta vs dense title: +0.0231
Search time: 1.77 s


4. Сохранение и таблица

In [105]:
np.savez(
    DENSE_DIR / "validation_title_params_embeddings.npz",
    item_embeddings=dense_params_embeddings,
    query_embeddings=dense_query_embeddings,
    item_codes=dense_params_codes,
    query_codes=dense_query_codes,
    item_ids=local_items["item_id"].to_numpy(dtype=str),
    query_ids=query_ids,
    model_name=np.array(DENSE_MODEL_NAME),
    max_seq_length=np.array(dense_model.max_seq_length),
    item_encoding_seconds=np.array(dense_params_encoding_time),
)

np.savez(
    DENSE_DIR / "validation_title_params_top50.npz",
    top_item_ids=local_items["item_id"].to_numpy(dtype=str)[
        dense_params_indices
    ],
    top_scores=dense_params_scores,
    query_codes=dense_query_codes,
    query_ids=query_ids,
    recall_at_50=np.array(dense_params_recall),
    search_seconds=np.array(dense_params_search_time),
)

In [95]:
dense_params_result = pd.DataFrame([{
    "model": "Multilingual E5-small",
    "item_fields": "title + params",
    "recall@50": dense_params_recall,
    "delta_vs_word": dense_params_recall - baseline_recall,
    "zero_queries_%": np.nan,
    # Здесь кодировались только объявления:
    # эмбеддинги запросов уже готовы.
    "vectorization_s": dense_params_encoding_time,
    "search_s": dense_params_search_time,
}])

replace_mask = (
    (experiments["model"] == "Multilingual E5-small")
    & (experiments["item_fields"] == "title + params")
)

experiments = pd.concat(
    [
        experiments.loc[~replace_mask],
        dense_params_result,
    ],
    ignore_index=True,
)

display(
    experiments
    .sort_values("recall@50", ascending=False)
    .round(4)
)

,model,item_fields,recall@50,zero_queries_%,vectorization_s,search_s,delta_vs_word
4,Multilingual E5-small,title + params,0.1836,NaN,168.2887,1.4900,0.0402
3,Multilingual E5-small,title,0.1605,NaN,47.6527,1.8144,0.0171
1,BM25,title,0.1515,0.2954,2.7702,54.9301,0.0080
2,BM25,title + params,0.1437,0.2293,40.0910,1569.3237,0.0002
0,Word TF-IDF,title,0.1435,0.2954,2.4670,54.3734,NaN


Добавление параметров повысило Recall@50 с 0.1605 до 0.1836 (+2,31 п.п.). В отличие от BM25, dense-представление использует дополнительный текст с улучшением качества.

## 5.6 Dense: Title + Params + Description

Объединим заголовок, параметры и описание объявления. Максимальная длина общего текста - 256 токенов. Запросы и способ поиска сохраняем прежними.

In [106]:
DESC_MAX_TOKENS = 256
DESC_BATCH_SIZE = 32
DESC_CHUNK_SIZE = 512

DESC_FIELDS = "title + params + description"

DESC_MODEL_LABEL = (
    f"Multilingual E5-small | doc={DESC_MAX_TOKENS}"
)

DESC_FILE_PREFIX = (
    f"validation_title_params_description_{DESC_MAX_TOKENS}"
)

dense_desc_texts = prepare_text(
    local_items["item_title_raw"].fillna("")
    + ". "
    + local_items["item_infm_params_text"].fillna("")
    + ". "
    + local_items["item_description_raw"].fillna("")
)

dense_desc_codes, dense_unique_desc = pd.factorize(
    dense_desc_texts,
    sort=False,
)

print(f"Unique texts: {len(dense_unique_desc):,}")
print(f"Document token limit: {DESC_MAX_TOKENS}")

del dense_desc_texts
gc.collect()

Unique texts: 344,787
Document token limit: 256


4

Получение эмбеддингов с прогрессом

In [107]:
dense_desc_embeddings = np.empty(
    (len(dense_unique_desc), dense_query_embeddings.shape[1]),
    dtype=np.float32,
)

previous_max_length = dense_model.max_seq_length
dense_model.max_seq_length = DESC_MAX_TOKENS

torch.cuda.synchronize()
started = perf_counter()

try:
    with tqdm_text(
        total=len(dense_unique_desc),
        desc=f"{DESC_FIELDS} | {DESC_MAX_TOKENS} tokens",
        unit="объявл.",
        file=sys.stdout,
        mininterval=1.0,
    ) as progress:

        for start in range(0, len(dense_unique_desc), DESC_CHUNK_SIZE):
            stop = min(
                start + DESC_CHUNK_SIZE,
                len(dense_unique_desc),
            )

            texts = [
                "passage: " + text
                for text in dense_unique_desc[start:stop]
            ]

            vectors = dense_model.encode(
                texts,
                batch_size=DESC_BATCH_SIZE,
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False,
            ).astype(np.float32)

            norms = np.linalg.norm(
                vectors,
                axis=1,
                keepdims=True,
            )

            if not np.isfinite(vectors).all() or (norms == 0).any():
                raise ValueError(
                    f"Invalid embeddings in rows {start}:{stop}"
                )

            vectors /= norms
            dense_desc_embeddings[start:stop] = vectors

            progress.update(stop - start)
            del texts, vectors, norms

finally:
    dense_model.max_seq_length = previous_max_length

torch.cuda.synchronize()
dense_desc_encoding_time = perf_counter() - started

print(f"Fields: {DESC_FIELDS}")
print(f"Embeddings: {dense_desc_embeddings.shape}")
print(f"Item encoding time: {dense_desc_encoding_time:.2f} s")

title + params + description | 256 tokens: 100%|██████████| 344787/344787 [09:52<00:00, 582.15объявл./s]
Fields: title + params + description
Embeddings: (344787, 384)
Item encoding time: 592.27 s


Сохранение эмбеддингов

In [108]:
np.savez(
    DENSE_DIR / f"{DESC_FILE_PREFIX}_embeddings.npz",
    item_embeddings=dense_desc_embeddings,
    query_embeddings=dense_query_embeddings,
    item_codes=dense_desc_codes,
    query_codes=dense_query_codes,
    item_ids=local_items["item_id"].to_numpy(dtype=str),
    query_ids=val_queries["query_id"].to_numpy(),
    model_name=np.array(DENSE_MODEL_NAME),
    item_fields=np.array(DESC_FIELDS),
    document_max_seq_length=np.array(DESC_MAX_TOKENS),
    query_max_seq_length=np.array(64),
    item_encoding_seconds=np.array(dense_desc_encoding_time),
)

print(f"Saved: {DESC_FILE_PREFIX}_embeddings.npz")

Saved: validation_title_params_description_256_embeddings.npz


Поиск и оценка

In [109]:
torch.cuda.synchronize()
started = perf_counter()

dense_desc_indices, dense_desc_scores = dense_top_k(
    unique_query_embeddings=dense_query_embeddings,
    unique_item_embeddings=dense_desc_embeddings,
    item_codes=dense_desc_codes,
    k=50,
    batch_size=128,
)

item_ids = local_items["item_id"].to_numpy()
query_ids = val_queries["query_id"].to_numpy()

desc_unique_predictions = item_ids[dense_desc_indices].tolist()

dense_desc_predictions = {
    query_id: desc_unique_predictions[int(text_code)]
    for query_id, text_code in zip(query_ids, dense_query_codes)
}

torch.cuda.synchronize()
dense_desc_search_time = perf_counter() - started

assert set(dense_desc_predictions) == set(val_relevant)
assert all(
    len(candidates) == len(set(candidates)) == 50
    for candidates in dense_desc_predictions.values()
)

dense_desc_recall = recall_at_k(
    dense_desc_predictions,
    val_relevant,
    k=50,
)

print(f"Dense title:                        {dense_recall:.4f}")
print(f"Dense title + params:               {dense_params_recall:.4f}")
print(f"Dense title + params + description: {dense_desc_recall:.4f}")

print(f"Delta vs title:  {dense_desc_recall - dense_recall:+.4f}")
print(f"Delta vs params: {dense_desc_recall - dense_params_recall:+.4f}")
print(f"Search time: {dense_desc_search_time:.2f} s")

Dense retrieval: 100%|██████████| 14884/14884 [00:01<00:00, 8973.40запрос/s]


Dense title:                        0.1605
Dense title + params:               0.1836
Dense title + params + description: 0.1778
Delta vs title:  +0.0172
Delta vs params: -0.0058
Search time: 1.98 s


Сохранение выдачи и обновление таблицы

In [110]:
np.savez(
    DENSE_DIR / f"{DESC_FILE_PREFIX}_top50.npz",
    top_item_ids=local_items["item_id"].to_numpy(dtype=str)[
        dense_desc_indices
    ],
    top_scores=dense_desc_scores,
    query_codes=dense_query_codes,
    query_ids=query_ids,
    model_name=np.array(DENSE_MODEL_NAME),
    item_fields=np.array(DESC_FIELDS),
    document_max_seq_length=np.array(DESC_MAX_TOKENS),
    recall_at_50=np.array(dense_desc_recall),
    search_seconds=np.array(dense_desc_search_time),
)

dense_desc_result = pd.DataFrame([{
    "model": DESC_MODEL_LABEL,
    "item_fields": DESC_FIELDS,
    "recall@50": dense_desc_recall,
    "delta_vs_word": dense_desc_recall - baseline_recall,
    "zero_queries_%": np.nan,
    "vectorization_s": dense_desc_encoding_time,
    "search_s": dense_desc_search_time,
}])

replace_mask = (
    (experiments["model"] == DESC_MODEL_LABEL)
    & (experiments["item_fields"] == DESC_FIELDS)
)

experiments = pd.concat(
    [
        experiments.loc[~replace_mask],
        dense_desc_result,
    ],
    ignore_index=True,
)

display(
    experiments
    .sort_values("recall@50", ascending=False)
    .round(4)
)

,model,item_fields,recall@50,zero_queries_%,vectorization_s,search_s,delta_vs_word
4,Multilingual E5-small,title + params,0.1836,NaN,168.2887,1.4900,0.0402
5,Multilingual E5-small | doc=256,title + params + description,0.1778,NaN,592.2718,1.9817,0.0343
3,Multilingual E5-small,title,0.1605,NaN,47.6527,1.8144,0.0171
1,BM25,title,0.1515,0.2954,2.7702,54.9301,0.0080
2,BM25,title + params,0.1437,0.2293,40.0910,1569.3237,0.0002
0,Word TF-IDF,title,0.1435,0.2954,2.4670,54.3734,NaN


Вариант title + params + description с лимитом 256 токенов снизил Recall@50 до 0.1778 (-0,58 п.п.), а кодирование заняло 592 с против 168 с.

# 6. Hybrid Retrieval

Объединим выдачи dense `title + params` и BM25 по заголовкам через Reciprocal Rank Fusion. Метод использует позиции кандидатов, поэтому не требует приводить оценки моделей к одному масштабу.

Проверим вес BM25 0.5 и 1.0 при фиксированном весе dense 1.0. Из объединённого списка выберем итоговые 50 объявлений.

функция объединения

In [111]:
def weighted_rrf(
    dense_items,
    sparse_items,
    sparse_weight=1.0,
    rrf_constant=60,
    k=50,
):
    scores = {}

    for rank, item_id in enumerate(dense_items[:k], start=1):
        scores[item_id] = 1.0 / (rrf_constant + rank)

    for rank, item_id in enumerate(sparse_items[:k], start=1):
        scores[item_id] = (
            scores.get(item_id, 0.0)
            + sparse_weight / (rrf_constant + rank)
        )

    return sorted(
        scores,
        key=scores.get,
        reverse=True,
    )[:k]

два эксперимента с прогрессом

In [112]:
assert set(dense_params_predictions) == set(val_relevant)
assert set(bm25_predictions) == set(val_relevant)

best_predictions = dense_params_predictions
best_recall = dense_params_recall
best_method = "Dense title + params"
best_sparse_weight = 0.0

rrf_results = [{
    "method": best_method,
    "bm25_weight": 0.0,
    "recall@50": best_recall,
    "delta_vs_dense": 0.0,
    "fusion_s": 0.0,
}]

for sparse_weight in [0.5, 1.0]:
    started = perf_counter()

    hybrid_predictions = {}

    for query_id in tqdm_text(
        val_relevant,
        desc=f"RRF | BM25 weight={sparse_weight}",
        unit="запрос",
        file=sys.stdout,
        mininterval=1.0,
    ):
        hybrid_predictions[query_id] = weighted_rrf(
            dense_items=dense_params_predictions[query_id],
            sparse_items=bm25_predictions[query_id],
            sparse_weight=sparse_weight,
            rrf_constant=60,
            k=50,
        )

    fusion_seconds = perf_counter() - started

    hybrid_recall = recall_at_k(
        hybrid_predictions,
        val_relevant,
        k=50,
    )

    rrf_results.append({
        "method": "Dense + BM25 RRF",
        "bm25_weight": sparse_weight,
        "recall@50": hybrid_recall,
        "delta_vs_dense": hybrid_recall - dense_params_recall,
        "fusion_s": fusion_seconds,
    })

    print(
        f"Recall@50: {hybrid_recall:.4f} | "
        f"Delta vs dense: {hybrid_recall - dense_params_recall:+.4f}",
        flush=True,
    )

    if hybrid_recall > best_recall:
        best_predictions = hybrid_predictions
        best_recall = hybrid_recall
        best_method = "Dense + BM25 RRF"
        best_sparse_weight = sparse_weight

    del hybrid_predictions

display(
    pd.DataFrame(rrf_results)
    .sort_values("recall@50", ascending=False)
    .round(4)
)

print(f"Selected method: {best_method}")
print(f"BM25 weight: {best_sparse_weight}")
print(f"Best validation Recall@50: {best_recall:.4f}")

RRF | BM25 weight=0.5: 100%|██████████| 74135/74135 [00:03<00:00, 20738.06запрос/s]
Recall@50: 0.1836 | Delta vs dense: +0.0000
RRF | BM25 weight=1.0: 100%|██████████| 74135/74135 [00:03<00:00, 23740.37запрос/s]
Recall@50: 0.1846 | Delta vs dense: +0.0010


,method,bm25_weight,recall@50,delta_vs_dense,fusion_s
2,Dense + BM25 RRF,1.0,0.1846,0.001,3.1357
0,Dense title + params,0.0,0.1836,0.000,0.0000
1,Dense + BM25 RRF,0.5,0.1836,0.000,3.5767


Selected method: Dense + BM25 RRF
BM25 weight: 1.0
Best validation Recall@50: 0.1846


сохранение выбранной конфигурации и validation-выдачи

In [113]:
import json

final_config = {
    "method": best_method,
    "dense_model": DENSE_MODEL_NAME,
    "dense_item_fields": [
        "item_title_raw",
        "item_infm_params_text",
    ],
    "dense_document_max_tokens": 64,
    "dense_query_max_tokens": 64,
    "bm25_item_fields": ["item_title_raw"],
    "bm25_k1": BM25_K1,
    "bm25_b": BM25_B,
    "dense_weight": 1.0,
    "bm25_weight": best_sparse_weight,
    "rrf_constant": 60,
    "source_top_k": 50,
    "output_top_k": 50,
    "validation_recall_at_50": float(best_recall),
}

with (DENSE_DIR / "selected_config.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(final_config, file, ensure_ascii=False, indent=2)

validation_ids = val_queries["query_id"].to_numpy()

assert all(
    len(best_predictions[query_id])
    == len(set(best_predictions[query_id]))
    == 50
    for query_id in validation_ids
)

np.savez(
    DENSE_DIR / "selected_validation_top50.npz",
    query_ids=validation_ids,
    item_ids=np.asarray(
        [best_predictions[query_id] for query_id in validation_ids],
        dtype=str,
    ),
)

print("Configuration and validation predictions saved")

Configuration and validation predictions saved


Объединение dense title + params и BM25 по заголовкам с равными весами повысило Recall@50 с 0.1836 до 0.1846 (+0,10 п.п.). Прирост небольшой, но объединение готовых выдач занимает около 3 секунд.

Попробуем сделать подбор параметров для RRF

In [114]:
DENSE_WEIGHTS = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]

rrf_tuning_rows = []

# best_predictions, best_recall и best_sparse_weight
# уже содержат лучший результат предыдущего эксперимента.

for dense_weight in DENSE_WEIGHTS:
    bm25_weight = 1.0 - dense_weight
    sparse_ratio = bm25_weight / dense_weight

    started = perf_counter()
    candidate_predictions = {}

    for query_id in tqdm_text(
        val_relevant,
        desc=f"Dense={dense_weight:.2f} | BM25={bm25_weight:.2f}",
        unit="запрос",
        file=sys.stdout,
        mininterval=1.0,
    ):
        candidate_predictions[query_id] = weighted_rrf(
            dense_items=dense_params_predictions[query_id],
            sparse_items=bm25_predictions[query_id],
            sparse_weight=sparse_ratio,
            rrf_constant=60,
            k=50,
        )

    fusion_seconds = perf_counter() - started

    score = recall_at_k(
        candidate_predictions,
        val_relevant,
        k=50,
    )

    rrf_tuning_rows.append({
        "dense_weight": dense_weight,
        "bm25_weight": bm25_weight,
        "recall@50": score,
        "delta_vs_dense": score - dense_params_recall,
        "fusion_s": fusion_seconds,
    })

    print(f"Recall@50: {score:.5f}", flush=True)

    if score > best_recall:
        best_predictions = candidate_predictions
        best_recall = score
        best_method = "Dense + BM25 RRF"
        best_sparse_weight = sparse_ratio

    del candidate_predictions

rrf_tuning_results = (
    pd.DataFrame(rrf_tuning_rows)
    .sort_values("recall@50", ascending=False)
    .reset_index(drop=True)
)

display(rrf_tuning_results.round(5))

selected_dense_weight = 1.0 / (1.0 + best_sparse_weight)
selected_bm25_weight = 1.0 - selected_dense_weight

print(f"Selected dense weight: {selected_dense_weight:.2f}")
print(f"Selected BM25 weight:  {selected_bm25_weight:.2f}")
print(f"Best Recall@50:       {best_recall:.5f}")

Dense=0.35 | BM25=0.65: 100%|██████████| 74135/74135 [00:01<00:00, 39511.57запрос/s]
Recall@50: 0.15578
Dense=0.40 | BM25=0.60: 100%|██████████| 74135/74135 [00:02<00:00, 27848.67запрос/s]
Recall@50: 0.16976
Dense=0.45 | BM25=0.55: 100%|██████████| 74135/74135 [00:01<00:00, 40692.75запрос/s]
Recall@50: 0.17841
Dense=0.50 | BM25=0.50: 100%|██████████| 74135/74135 [00:01<00:00, 39863.26запрос/s]
Recall@50: 0.18457
Dense=0.55 | BM25=0.45: 100%|██████████| 74135/74135 [00:01<00:00, 41502.96запрос/s]
Recall@50: 0.18610
Dense=0.60 | BM25=0.40: 100%|██████████| 74135/74135 [00:01<00:00, 37915.54запрос/s]
Recall@50: 0.18559
Dense=0.65 | BM25=0.35: 100%|██████████| 74135/74135 [00:02<00:00, 32141.03запрос/s]
Recall@50: 0.18361


,dense_weight,bm25_weight,recall@50,delta_vs_dense,fusion_s
0,0.55,0.45,0.18610,0.00250,1.78889
1,0.60,0.40,0.18559,0.00198,1.95605
2,0.50,0.50,0.18457,0.00096,1.86223
3,0.65,0.35,0.18361,0.00000,2.30829
4,0.45,0.55,0.17841,-0.00519,1.82394
5,0.40,0.60,0.16976,-0.01385,2.66343
6,0.35,0.65,0.15578,-0.02782,1.87886


Selected dense weight: 0.55
Selected BM25 weight:  0.45
Best Recall@50:       0.18610


обновление сохранённого результата

In [115]:
final_config.update({
    "method": best_method,
    "dense_weight": 1.0,
    "bm25_weight": float(best_sparse_weight),
    "rrf_constant": 60,
    "validation_recall_at_50": float(best_recall),
})

with (DENSE_DIR / "selected_config.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(final_config, file, ensure_ascii=False, indent=2)

validation_ids = val_queries["query_id"].to_numpy()

assert all(
    len(best_predictions[query_id])
    == len(set(best_predictions[query_id]))
    == 50
    for query_id in validation_ids
)

np.savez(
    DENSE_DIR / "selected_validation_top50.npz",
    query_ids=validation_ids,
    item_ids=np.asarray(
        [best_predictions[query_id] for query_id in validation_ids],
        dtype=str,
    ),
)

rrf_tuning_results.to_csv(
    DENSE_DIR / "rrf_weight_search.csv",
    index=False,
)

print("Best configuration and predictions saved")

Best configuration and predictions saved


Лучший результат получен при весах dense 0.55 и BM25 0.45: Recall@50 = 0.18610. Это на 0,15 п.п. выше RRF с равными весами и на 0,25 п.п. выше отдельного dense.

# 7. Final

In [117]:
equal_weights_recall = rrf_tuning_results.loc[
    np.isclose(rrf_tuning_results["dense_weight"], 0.5),
    "recall@50",
].iloc[0]

final_results = experiments[[
    "model",
    "item_fields",
    "recall@50",
]].copy()

hybrid_results = pd.DataFrame([
    {
        "model": "RRF | dense=0.50, BM25=0.50",
        "item_fields": "Dense: title + params; BM25: title",
        "recall@50": equal_weights_recall,
    },
    {
        "model": (
            f"RRF | dense={selected_dense_weight:.2f}, "
            f"BM25={selected_bm25_weight:.2f}"
        ),
        "item_fields": "Dense: title + params; BM25: title",
        "recall@50": best_recall,
    },
])

final_results = pd.concat(
    [final_results, hybrid_results],
    ignore_index=True,
)

final_results["delta_vs_baseline_pp"] = (
    final_results["recall@50"] - baseline_recall
) * 100

final_results = (
    final_results
    .sort_values("recall@50", ascending=False)
    .reset_index(drop=True)
    .rename(columns={
        "model": "Метод",
        "item_fields": "Текст объявления",
        "recall@50": "Recall@50",
        "delta_vs_baseline_pp": "Прирост к Word TF-IDF, п.п.",
    })
)

display(
    final_results.style
    .format({
        "Recall@50": "{:.5f}",
        "Прирост к Word TF-IDF, п.п.": "{:+.2f}",
    })
    .hide(axis="index")
)

final_results.to_csv(
    DENSE_DIR / "final_experiments.csv",
    index=False,
)

Метод,Текст объявления,Recall@50,"Прирост к Word TF-IDF, п.п."
"RRF | dense=0.55, BM25=0.45",Dense: title + params; BM25: title,0.18610,+4.26
"RRF | dense=0.50, BM25=0.50",Dense: title + params; BM25: title,0.18457,+4.11
Multilingual E5-small,title + params,0.18362,+4.02
Multilingual E5-small | doc=256,title + params + description,0.17776,+3.43
Multilingual E5-small,title,0.16054,+1.71
BM25,title,0.15146,+0.80
BM25,title + params,0.14370,+0.02
Word TF-IDF,title,0.14345,+0.00


# 8. Submission

Применим выбранную конфигурацию к benchmark: dense по заголовкам и параметрам с лимитом 64 токена, BM25 по заголовкам, затем RRF с подобранными весами.

конфигурация и данные

In [118]:
with (DENSE_DIR / "selected_config.json").open(
    encoding="utf-8"
) as file:
    submission_config = json.load(file)

assert DENSE_MODEL_NAME == submission_config["dense_model"]
assert queries["query_id"].is_unique
assert items["item_id"].is_unique
assert len(items) >= 50

bench_query_ids = queries["query_id"].to_numpy()
bench_item_ids = items["item_id"].to_numpy()

bench_query_texts = prepare_text(queries["search_query"])
bench_titles = prepare_text(items["item_title_raw"])

bench_dense_texts = prepare_text(
    items["item_title_raw"].fillna("")
    + ". "
    + items["item_infm_params_text"].fillna("")
)

bench_item_codes, bench_unique_items = pd.factorize(
    bench_dense_texts,
    sort=False,
)

bench_query_codes, bench_unique_queries = pd.factorize(
    bench_query_texts,
    sort=False,
)

print(f"Benchmark queries: {len(queries):,}")
print(f"Benchmark items: {len(items):,}")
print(f"Unique dense item texts: {len(bench_unique_items):,}")

print(
    "Normalized RRF weights:",
    f"dense={1 / (1 + submission_config['bm25_weight']):.2f},",
    f"BM25={submission_config['bm25_weight'] / (1 + submission_config['bm25_weight']):.2f}",
)

Benchmark queries: 2,452
Benchmark items: 189,212
Unique dense item texts: 188,986
Normalized RRF weights: dense=0.55, BM25=0.45


эмбеддинги benchmark с прогрессом

In [119]:
def encode_benchmark(texts, prefix, max_tokens, label):
    embeddings = np.empty(
        (len(texts), dense_query_embeddings.shape[1]),
        dtype=np.float32,
    )

    previous_length = dense_model.max_seq_length
    dense_model.max_seq_length = max_tokens

    try:
        with tqdm_text(
            total=len(texts),
            desc=label,
            unit="текст",
            file=sys.stdout,
            mininterval=1.0,
        ) as progress:

            for start in range(0, len(texts), 512):
                stop = min(start + 512, len(texts))

                vectors = dense_model.encode(
                    [prefix + text for text in texts[start:stop]],
                    batch_size=64,
                    normalize_embeddings=True,
                    convert_to_numpy=True,
                    show_progress_bar=False,
                ).astype(np.float32)

                norms = np.linalg.norm(
                    vectors, axis=1, keepdims=True
                )

                if not np.isfinite(vectors).all() or (norms == 0).any():
                    raise ValueError("Invalid benchmark embeddings")

                embeddings[start:stop] = vectors / norms

                progress.update(stop - start)
                del vectors, norms

    finally:
        dense_model.max_seq_length = previous_length

    return embeddings


bench_item_embeddings = encode_benchmark(
    bench_unique_items,
    prefix="passage: ",
    max_tokens=submission_config["dense_document_max_tokens"],
    label="Benchmark items",
)

bench_query_embeddings = encode_benchmark(
    bench_unique_queries,
    prefix="query: ",
    max_tokens=submission_config["dense_query_max_tokens"],
    label="Benchmark queries",
)

Benchmark queries: 100%|██████████| 2451/2451 [00:00<00:00, 2856.98текст/s]


dense top-50

In [120]:
bench_dense_indices, bench_dense_scores = dense_top_k(
    unique_query_embeddings=bench_query_embeddings,
    unique_item_embeddings=bench_item_embeddings,
    item_codes=bench_item_codes,
    k=submission_config["source_top_k"],
    batch_size=128,
)

bench_unique_dense_predictions = (
    bench_item_ids[bench_dense_indices].tolist()
)

bench_dense_predictions = {
    query_id: bench_unique_dense_predictions[int(text_code)]
    for query_id, text_code in zip(
        bench_query_ids, bench_query_codes
    )
}

print(f"Dense predictions: {len(bench_dense_predictions):,}")

Dense retrieval: 100%|██████████| 2451/2451 [00:00<00:00, 14303.64запрос/s]


Dense predictions: 2,452


BM25 top-50

In [121]:
print("Building benchmark BM25...", flush=True)

bench_bm25_items, bench_bm25_queries = build_bm25_vectors(
    item_texts=bench_titles,
    query_texts=bench_query_texts,
    k1=submission_config["bm25_k1"],
    b=submission_config["bm25_b"],
)

print("Retrieving BM25 candidates...", flush=True)

bench_bm25_predictions = retrieve_top_k(
    query_vectors=bench_bm25_queries,
    item_vectors=bench_bm25_items,
    query_ids=bench_query_ids,
    item_ids=bench_item_ids,
    k=submission_config["source_top_k"],
    batch_size=64,
)

print(f"BM25 predictions: {len(bench_bm25_predictions):,}")

Building benchmark BM25...
Retrieving BM25 candidates...
BM25 predictions: 2,452


объединение выдач

In [122]:
sparse_ratio = (
    submission_config["bm25_weight"]
    / submission_config["dense_weight"]
)

bench_predictions = {
    query_id: weighted_rrf(
        dense_items=bench_dense_predictions[query_id],
        sparse_items=bench_bm25_predictions[query_id],
        sparse_weight=sparse_ratio,
        rrf_constant=submission_config["rrf_constant"],
        k=submission_config["output_top_k"],
    )
    for query_id in bench_query_ids
}

проверка и сохранение answer.csv

In [123]:
valid_item_ids = set(bench_item_ids)

assert set(bench_predictions) == set(bench_query_ids)

for query_id, candidates in bench_predictions.items():
    assert len(candidates) == 50, query_id
    assert len(set(candidates)) == 50, query_id
    assert set(candidates).issubset(valid_item_ids), query_id

answer = pd.DataFrame({
    "query_id": bench_query_ids,
    "answer": [
        " ".join(bench_predictions[query_id])
        for query_id in bench_query_ids
    ],
})

assert answer.columns.tolist() == ["query_id", "answer"]
assert len(answer) == len(queries)
assert answer["query_id"].is_unique
assert answer["query_id"].str.len().eq(16).all()

ANSWER_PATH = Path("answer.csv").resolve()

answer.to_csv(
    ANSWER_PATH,
    index=False,
    encoding="utf-8",
)

# Проверяем файл после сохранения, сохраняя ID строками.
saved_answer = pd.read_csv(
    ANSWER_PATH,
    dtype=str,
    keep_default_na=False,
)

pd.testing.assert_frame_equal(
    saved_answer,
    answer.astype(str),
    check_dtype=False,
)

print(f"Saved: {ANSWER_PATH}")
print(f"Queries: {len(saved_answer):,}")
print("Candidates per query: 50")
print("All checks passed")

display(saved_answer.head())

Saved: C:\Users\alesh\Documents\Apps and Data\GITHUB\nlp-search-candidate-generation\answer.csv
Queries: 2,452
Candidates per query: 50
All checks passed


,query_id,answer
0,70DfDUpwjxB4lzFd,fa01652ca5f57d0f d722bcda1a555091 603623b4bd9e...
1,JTrdTaZJvSiLPkXj,4a435cc0a254c1fd 16ea26ff0aa0450a 8703b2a0442c...
2,LZCZNoVG4AFUkVRJ,dab52187b4500d9b 7347ccc4bdcfde01 d62074dd39ca...
3,660ac9QVtXkRxZC3,f9028eecc17d9dd0 250f823c8918c2d8 130544bc3ab3...
4,YgHcM9MVbxKnxD1e,c539e0661217b8eb 6512376fc63b409e 23bc4a9372c1...
